In [1]:
language = "english"

dump_file = f"/mnt/Velocity_Vault/Wiki_Dump/Dump/{language}_dump.xml"
raw_corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_corpus.txt"
corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_processed.txt"

co_occur_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_matrix.npy"
pearson_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_pearson_matrix.npy"
svd_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_svd_matrix.npy"

In [2]:
import numpy as np

def compute_pearson_correlation(input_matrix_file, output_matrix_file):
    
    """Compute Pearson correlation matrix with proper implementation"""
    
    X = np.load(input_matrix_file).astype(np.float32)
    n = X.shape[0]
    
    print(f"Computing Pearson correlation for {n}x{n} matrix...")
    
    row_means = np.mean(X, axis=1, keepdims=True)
    X_centered = X - row_means
    
    variances = np.var(X_centered, axis=1, ddof=1)
    
    variance_threshold = 1e-4
    low_variance_mask = variances < variance_threshold
    
    stds = np.sqrt(variances)
    stds[low_variance_mask] = 1.0
    
    X_normalized = X_centered / stds[:, np.newaxis]
    
    del X, X_centered
    
    print("Computing Pearson correlation matrix")
    
    corr_matrix = (X_normalized @ X_normalized.T) / (n - 1)
    
    del X_normalized
    
    corr_matrix[low_variance_mask, :] = 0.0
    corr_matrix[:, low_variance_mask] = 0.0
    np.fill_diagonal(corr_matrix, 1.0)
    
    np.save(output_matrix_file, corr_matrix)
    
    print("Pearson correlation computation complete")

compute_pearson_correlation(co_occur_matrix_file, pearson_matrix_file)

Computing Pearson correlation for 22590x22590 matrix...
Computing Pearson correlation matrix
Pearson correlation computation complete


In [3]:
from collections import Counter

def get_vocabulary(filename, min_freq):
    """Reads corpus file, counts word occurrences.
    Returns a list of words sorted by frequency (highest first)."""
    
    with open(filename, 'r') as file:
        words = file.read().strip().split()
        
    freq_dict = Counter(words)
    
    word_freq_pairs = [(word, freq) for word, freq in freq_dict.items() if freq >= min_freq]
    
    sorted_word_freq_pairs = sorted(word_freq_pairs, key=lambda x: x[1], reverse=True)
    
    vocab_list = [word for word, _ in sorted_word_freq_pairs]
    
    return vocab_list

vocabulary = get_vocabulary(corpus_file, 250)
print("Vocabulary Size - ", len(vocabulary))

Vocabulary Size -  22590


In [4]:

import numpy as np
from tabulate import tabulate

def display_similar_words(pcm_file, cooccur_file, vocab_list, target_words):
    '''Display top 5 similar words with cosine similarity from co-occurrence matrix. Each target word 
    is a column header with all 5 results in one cell.'''
    n = len(vocab_list)
    
    P = np.load(pcm_file).astype(np.float32)
    C = np.load(cooccur_file).astype(np.int32)
    
    results = []
    
    for target_word in target_words:
        target_idx = vocab_list.index(target_word)
        pearson_scores = P[target_idx, :]
        top_indices = np.argsort(pearson_scores)[::-1][1:6]
        target_vector = C[target_idx, :].astype(np.float32)
        similar_lines = []
        
        for idx in top_indices:
            similar_word = vocab_list[idx]
            similar_vector = C[idx, :].astype(np.float32)
            cosine_sim = np.dot(target_vector, similar_vector) / (np.linalg.norm(target_vector) * np.linalg.norm(similar_vector))
            similar_lines.append(f"{similar_word} ({cosine_sim:.4f})")
        
        results.append("\n".join(similar_lines))
    
    table_data = [results]
    print(tabulate(table_data, headers=target_words, tablefmt="grid"))

target_words = vocabulary[460:465]
display_similar_words(pearson_matrix_file, co_occur_matrix_file, vocabulary, target_words)

+------------------------+----------------------+-----------------------+------------------+------------------------+
| we                     | uk                   | books                 | southern         | airport                |
+========================+======================+=======================+==================+========================+
| you (0.8942)           | netherlands (0.9450) | works (0.9333)        | eastern (0.9811) | airstrip (0.7461)      |
| they (0.8912)          | balkans (0.9401)     | publications (0.9276) | western (0.9737) | airfield (0.7414)      |
| so (0.8556)            | hague (0.9368)       | articles (0.9211)     | fall (0.9703)    | terminal (0.7400)      |
| things (0.8442)        | ninth (0.9333)       | papers (0.9164)       | region (0.9697)  | amnesty (0.7345)       |
| automatically (0.8420) | midwest (0.9331)     | poems (0.9146)        | waters (0.9692)  | international (0.7269) |
+------------------------+----------------------+-------